# Day 19 — String Manipulation & **Text Processing**

**Phase 1 · Module 3: Prompt Engineering & RAG** · ~30 min

The afternoon's topic is prompt engineering; the plumbing under it is **text**. Two jobs recur constantly at a bank:

1. **Extract** structured facts out of messy free text — amounts, dates, invoice numbers from a statement or email — with the `re` module.
2. **Build** clean prompts — multi-line templates without ragged indentation, and safe variable substitution — with `textwrap.dedent`, `str.format_map`, and f-strings.

Plus the step every RAG pipeline needs before embedding: **normalise** text so the same words match regardless of case or punctuation.

> **Runnable keyless:** pure standard library — `re`, `textwrap`, `string`. No API key, no installs.

## 1. `re` — extract amounts, dates, invoice numbers

A regex is a small pattern language for matching text. Three building blocks do most bank work: `\d` (a digit), quantifiers (`+` one-or-more, `{2}` exactly two), and **groups** `( )` to capture the piece you want. `re.findall` returns every match.

In [1]:
import re

statement = """Invoice INV-2024-0917 issued 14/03/2024 for GBP 1,299.50.
Late fee of £45.00 applied on 2024-04-01. Ref INV-2024-1002, amount 12,000.00."""

# money: optional £/GBP, digits with thousands separators, 2 decimal places
amounts = re.findall(r'(?:£|GBP\s?)?(\d{1,3}(?:,\d{3})*\.\d{2})', statement)
print("amounts :", amounts)

# invoice ids: INV-YYYY-NNNN
invoices = re.findall(r'INV-\d{4}-\d{4}', statement)
print("invoices:", invoices)

# dates in two formats: DD/MM/YYYY or YYYY-MM-DD
dates = re.findall(r'\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2}', statement)
print("dates   :", dates)

amounts : ['1,299.50', '45.00', '12,000.00']
invoices: ['INV-2024-0917', 'INV-2024-1002']
dates   : ['14/03/2024', '2024-04-01']


☝ `(?:...)` is a **non-capturing** group (match but don't return it), so `findall` yields only the number inside the capturing `( )`. The `|` alternation matches either date format. This is exactly how you'd pull entities out of a customer email before handing them to an agent.

In [2]:
# Named groups + finditer: get each match WITH its fields and position
pattern = re.compile(r'(?P<id>INV-\d{4}-\d{4}).*?(?P<amt>\d[\d,]*\.\d{2})', re.DOTALL)
for m in pattern.finditer(statement):
    print(f"{m.group('id')}  ->  {m.group('amt'):>10}   (chars {m.start()}-{m.end()})")

INV-2024-0917  ->    1,299.50   (chars 8-56)
INV-2024-1002  ->   12,000.00   (chars 104-135)


☝ `(?P<name>...)` names a group so you read `m.group('id')` instead of counting positions. `finditer` gives match **objects** (position, groups) rather than plain strings — use it when you need more than the text. `re.DOTALL` lets `.` cross newlines.

## 2. `re.sub` — clean and redact

`re.sub(pattern, replacement, text)` rewrites every match — the workhorse for redaction and normalisation. The replacement can reference captured groups (`\1`) or call a function.

In [3]:
note = "Call cust on 07700 900123 or email jo.smith@barclays.co.uk about acct 12345678."

redacted = re.sub(r'\b\d{8}\b', '[ACCT]', note)                     # 8-digit account
redacted = re.sub(r'\b0\d{4}\s?\d{6}\b', '[PHONE]', redacted)       # UK mobile
redacted = re.sub(r'\b[\w.]+@[\w.]+\b', '[EMAIL]', redacted)        # email
print(redacted)

# function replacement: mask all but the last 4 digits of any long number
mask = re.sub(r'\d{6,}', lambda m: '*' * (len(m.group()) - 4) + m.group()[-4:], note)
print(mask)

Call cust on [PHONE] or email [EMAIL] about acct [ACCT].
Call cust on 07700 **0123 or email jo.smith@barclays.co.uk about acct ****5678.


☝ A **function** as the replacement gives full control per match — here keeping the last four digits, the standard way to show a card or account safely. This is the regex layer under the Day-16/18 PII guardrails.

## 3. `textwrap.dedent` — clean multi-line prompt templates

Prompts written as triple-quoted strings inside a function inherit the code's indentation — every line starts with spaces the model shouldn't see. `textwrap.dedent` strips the **common leading whitespace**, so the template reads cleanly.

In [4]:
import textwrap

def build_prompt_bad(question):
    return """
        You are a loan-policy assistant.
        Answer only from the policy.
        Question: """ + question

def build_prompt(question):
    tpl = """
        You are a loan-policy assistant.
        Answer only from the policy.
        Question: {q}
    """
    return textwrap.dedent(tpl).strip().format(q=question)

print("--- ragged (note leading spaces) ---")
print(repr(build_prompt_bad("max LTV?")[:60]))
print("--- dedented ---")
print(build_prompt("max LTV?"))

--- ragged (note leading spaces) ---
'\n        You are a loan-policy assistant.\n        Answer onl'
--- dedented ---
You are a loan-policy assistant.
Answer only from the policy.
Question: max LTV?


☝ `dedent` removes the indentation **all lines share**, turning source-readable templates into clean prompt text. `.strip()` drops the leading/trailing blank lines from the triple-quote. Ragged indentation wastes tokens and can confuse the model.

## 4. `str.format_map` vs f-strings — substituting into prompts

Both fill placeholders. The difference matters when the values live in a **dict** (a prompt's variables often do) and when the template comes from *outside* your code (a YAML store — this afternoon's topic).

In [5]:
vars = {"role": "fraud analyst", "channel": "mobile app"}

# f-string: template is in the source, values are local names — great for fixed prompts
role, channel = vars["role"], vars["channel"]
print(f"You are a {role} reviewing a {channel} transaction.")

# format_map: template is DATA, values come from a dict — great for stored/loaded templates
template = "You are a {role} reviewing a {channel} transaction."
print(template.format_map(vars))

You are a fraud analyst reviewing a mobile app transaction.
You are a fraud analyst reviewing a mobile app transaction.


In [6]:
# format_map's edge: a missing key. Default it instead of crashing, with a dict subclass.
class Safe(dict):
    def __missing__(self, key):
        return "{" + key + "}"          # leave unknown placeholders untouched

partial = "You are a {role} in {region}."
print(partial.format_map(Safe(vars)))   # region unknown -> left as-is, no KeyError

You are a fraud analyst in {region}.


☝ Rule of thumb: **f-string** when the template is literal in your code; **`format_map`** when the template is *data* (loaded from YAML/DB) and the values are a dict. `format_map` doesn't copy the dict (unlike `**kwargs`), and a `__missing__` subclass makes partial fills safe — essential when prompt templates come from a store.

## 5. Normalise text before embedding

Retrieval matches *tokens*. `"Buy-To-Let!"` and `"buy to let"` should hit the same chunk, so before embedding (or lexical scoring, as in yesterday's KB) you normalise: lowercase, strip, collapse whitespace, drop punctuation.

In [7]:
def normalise(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)      # punctuation -> space
    text = re.sub(r'\s+', ' ', text)          # collapse runs of whitespace
    return text.strip()

samples = ["  Buy-To-Let Mortgage!! ", "buy   to let\tmortgage", "BUY-TO-LET mortgage."]
for s in samples:
    print(f"{s!r:35} -> {normalise(s)!r}")

print("all identical:", len({normalise(s) for s in samples}) == 1)

'  Buy-To-Let Mortgage!! '          -> 'buy to let mortgage'
'buy   to let\tmortgage'            -> 'buy to let mortgage'
'BUY-TO-LET mortgage.'              -> 'buy to let mortgage'
all identical: True


☝ Three superficially different strings collapse to one canonical form, so they retrieve together. This is the pre-processing step in front of every embedding model and yesterday's `_terms()` KB scorer — consistent normalisation is what makes retrieval reliable.

## 6. Recap — the text layer under every prompt

| Tool | Why used |
|---|---|
| `re.findall` / `finditer` | **extract** amounts, dates, invoice ids from free text |
| named groups `(?P<n>…)` | read matches by name, not position |
| `re.sub` (+ func replace) | **redact / normalise** — the layer under PII guardrails |
| `textwrap.dedent` | strip source indentation from multi-line **prompt templates** |
| f-string | substitute when the template is **literal in code** |
| `str.format_map` (+ `__missing__`) | substitute when the template is **data** (YAML store) |
| `normalise()` | canonicalise text so retrieval matches regardless of case/punctuation |


